# Lopez de Prado Architecture: Validation Walkthrough

This notebook demonstrates the full pipeline refactoring based on "Advances in Financial Machine Learning".

**Pipeline Steps:**
1.  **Dollar Bars**: Sampling ticks by volume.
2.  **Primary Model**: Generating signals (e.g., Moving Average Crossover).
3.  **Fractional Differentiation**: Preserving memory while achieving stationarity.
4.  **Triple Barrier Method**: Labelling events with Proft-Take, Stop-Loss, and Time Barriers.
5.  **Meta-Labeling**: Using ML to filter Primary Signals.
6.  **Purged K-Fold CV**: Validating without leakage.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

from sampling.dollar_bars import get_dollar_bars
from data_handler import DataHandler
from strategies.primary_models import MovingAverageCrossover, PrimaryModel
from labeling.barriers import get_triple_barrier_events, get_bins, get_volatility
from features.frac_diff import frac_diff_ffd
from ml.meta_labeler import MetaLabeler
from backtest.cross_validation import PurgedKFold

%matplotlib inline

## 1. Load Data & Generate Dollar Bars
We will fetch 1-minute bars (or ticks if available) and construct Dollar Bars.

In [ ]:
# For this demo, we can generate synthetic data if API keys are missing, 
# or try to fetch real data.

def generate_synthetic_trades(n=10000):
    dates = pd.date_range(start='2024-01-01', periods=n, freq='S')
    price = 100 + np.cumsum(np.random.normal(0, 0.05, n))
    size = np.random.randint(1, 100, n)
    return pd.DataFrame({'price': price, 'size': size}, index=dates)

try:
    trades = generate_synthetic_trades(n=50000)
    print(f"Generated {len(trades)} synthetic trades.")
    
    # Dollar Bars
    # Threshold: average trade is ~50 * 100 = 5000. 
    # Let's say we want a bar every 50 trades -> 250,000
    dollar_bars = get_dollar_bars(trades, threshold=250000)
    print(f"Generated {len(dollar_bars)} Dollar Bars.")
    
    plt.figure(figsize=(10, 5))
    plt.plot(dollar_bars['close'])
    plt.title("Dollar Bars (Synthetic)")
    plt.show()
    
except Exception as e:
    print(f"Error: {e}")

## 2. Run Primary Model
Here we plug in our strategy. 
**Modularity**: Change `MovingAverageCrossover` to any class inheriting from `PrimaryModel`.

In [ ]:
# Instantiate Strategy
strategy = MovingAverageCrossover(fast_window=10, slow_window=50)

# Generate Signals
events = strategy.generate_signals(dollar_bars['close'])
print(f"Generated {len(events)} signals.")
print(events.head())

## 3. Triple Barrier Labeling
Determine if the signals were successful.

In [ ]:
vol = get_volatility(dollar_bars['close'], span=50)

# Vertical barrier: 50 bars later
vertical_barriers = events.index + pd.Timedelta(minutes=50) # Approx time, strictly should use bar index logic
# Or better, map index to time if we have many bars per minute.
vertical_barriers = pd.Series(vertical_barriers, index=events.index)

barrier_events = get_triple_barrier_events(
    close=dollar_bars['close'],
    t_events=events.index,
    pt_sl=[1, 1],
    target=vol,
    min_ret=0.0001,
    vertical_barrier_times=vertical_barriers,
    side=events['side']
)

labels = get_bins(barrier_events, dollar_bars['close'])
print("Label Distribution:")
print(labels['bin'].value_counts())

## 4. Feature Extraction & FracDiff
We calculate features for the Meta-Labeler.

In [ ]:
# 1. Fractional Diff
log_prices = np.log(dollar_bars['close'])
frac_diff = frac_diff_ffd(log_prices, d=0.4)
frac_diff.name = 'frac_diff'

# 2. Align Features
features = pd.DataFrame(index=labels.index)
features['volatility'] = vol.loc[labels.index]
features['side'] = events.loc[labels.index, 'side']
features['frac_diff'] = frac_diff.loc[labels.index]

# Drop NaNs
features = features.dropna()
y = labels['bin'].loc[features.index]

print(f"Training Data: {features.shape}")

## 5. Purged K-Fold Cross Validation
Validate the Meta-Labeler.

In [ ]:
metrics = []

# Define t1 for purging (End time of the barriers)
# Use the t1 from labels/barrier_events
t1_series = barrier_events['t1']
t1_aligned = t1_series.loc[features.index]

cv = PurgedKFold(n_splits=5, t1=t1_aligned, pct_embargo=0.01)

model = MetaLabeler()

for train_idx, test_idx in cv.split(features):
    X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    model.fit(X_train, y_train)
    probs = model.predict(X_test)
    
    # Store metrics (AUC, Accuracy)
    from sklearn.metrics import roc_auc_score, accuracy_score
    try:
        auc = roc_auc_score(y_test, probs)
    except:
        auc = 0.5
    acc = accuracy_score(y_test, (probs > 0.5).astype(int))
    metrics.append({'auc': auc, 'acc': acc})
    
print("\nAverage Performance:")
print(pd.DataFrame(metrics).mean())